# Positional encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.d_model = d_model
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

# Feed-forward network

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForwardNetwork, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

# Masked multi-head self-attention

Reference [QA1](./QA1_Decode.ipynb.md) for this line code:
```python
mask = torch.triu(torch.ones(x.size(0), x.size(0), device=x.device), diagonal=1).bool()
```

In [ ]:
class MaskedMultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1):
        super(MaskedMultiHeadSelfAttention, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mask = torch.triu(torch.ones(x.size(0), x.size(0), device=x.device), diagonal=1).bool() # Note: this code
        attn_output, _ = self.self_attn(x, x, x, attn_mask=mask)
        x = x + self.dropout(attn_output)
        x = self.norm(x)
        return x

# Transformer decoder block

In [ ]:
class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, nhead, d_ff, dropout=0.1):
        super(TransformerDecoderBlock, self).__init__()
        self.self_attn = MaskedMultiHeadSelfAttention(d_model, nhead, dropout)
        self.feed_forward = FeedForwardNetwork(d_model, d_ff, dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.self_attn(x)
        x = x + self.feed_forward(x)
        return x

# Transformer decoder model

In [ ]:
class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, nhead, d_ff, dropout=0.1):
        super(TransformerDecoder, self).__init__()
        self.layers = nn.ModuleList([TransformerDecoderBlock(d_model, nhead, d_ff, dropout) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        return x

# Example usage of the transformer decoder

In [ ]:
if __name__ == "__main__":
    d_model = 512
    nhead = 8
    d_ff = 2048
    num_layers = 6
    seq_length = 10
    batch_size = 2

    decoder = TransformerDecoder(num_layers, d_model, nhead, d_ff, dropout=0.1)
    x = torch.rand(seq_length, batch_size, d_model)
    output = decoder(x)
    print(output.shape)